# Silver → Gold Iceberg Pipeline (DataFrame + SQL Hybrid)

Hybrid approach using DataFrame API for transformations and SQL for MERGE / DDL.

## 0. Spark & Iceberg configuration

In [18]:
%%configure -f
{
  "conf": {
    "spark.ui.enabled": "true",
    "spark.ui.port": "4040",

    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.glue_catalog": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.glue_catalog.catalog-impl": "org.apache.iceberg.aws.glue.GlueCatalog",
    "spark.sql.catalog.glue_catalog.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    "spark.sql.catalog.glue_catalog.warehouse": "s3://dummy-lakehouse"
  }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,None,pyspark,idle,,,None,✔


In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Args, Globals & Imports

In [20]:
import sys, json
from pyspark.sql.functions import *
from pyspark.sql.window import Window

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
sys.argv = [
    "glue_job_entrypoint.py",   # always present, ignore
    "--domain=american_markets",
    "--source=alphavantage",
    "--dataset=time_series_daily",
    "--keys=[\"bronze/american_markets/source=alphavantage/dataset=time_series_daily/ingestion_date=2025-12-19/time_series_daily_AAPL_20251219_185817.jsonl\","
            "\"bronze/american_markets/source=alphavantage/dataset=time_series_daily/ingestion_date=2025-12-19/time_series_daily_MSFT_20251219_185817.json\"]",
    "--record_count=2",
    "--ingested_at=2026-01-08T18:58:17Z",
    "--dag_id=tariff_lakehouse_dag",
    "--run_id=scheduled__2026-01-08T18:59:00+00:00",
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
def parse_args(argv):
    args = {}
    for arg in argv[1:]:               # skip argv[0]
        if not arg.startswith("--"):
            continue
        key, value = arg[2:].split("=", 1)
        args[key] = value
    return args


ARGS = parse_args(sys.argv)

DOMAIN = ARGS["domain"]
SOURCE = ARGS["source"]
DATASET = ARGS["dataset"]
KEYS = json.loads(ARGS["keys"]) # list of relative paths/keys to ingested data files
RECORD_COUNT = int(ARGS["record_count"])
INGESTED_AT = ARGS["ingested_at"]
DAG_ID = ARGS["dag_id"]
RUN_ID = ARGS["run_id"]

if not KEYS:
    raise ValueError("No bronze files provided to Silver job")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
GLUE_CATALOG = "glue_catalog"
WAREHOUSE = "s3://dummy-lakehouse"
SILVER_DB = "silver"
GOLD_DB = "gold"

SILVER_TABLE = f"{DOMAIN}_{SOURCE}_{DATASET}_clean"
GOLD_TABLE = f"{DOMAIN}_{SOURCE}_{DATASET}_fact"
METRICS_TABLE = f"{DOMAIN}_{SOURCE}_{DATASET}_metrics"

SILVER_FQN = f"{GLUE_CATALOG}.{SILVER_DB}.{SILVER_TABLE}"
GOLD_FQN = f"{GLUE_CATALOG}.{GOLD_DB}.{GOLD_TABLE}"
METRICS_FQN = f"{GLUE_CATALOG}.{GOLD_DB}.{METRICS_TABLE}"

BRONZE_PATHS = [f"{WAREHOUSE}/{k}" for k in KEYS]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
print(f"Spark session started, configured to catalog={GLUE_CATALOG} and lakehouse_bucket={spark.conf.get(f'spark.sql.catalog.{GLUE_CATALOG}.warehouse')}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark session started, configured to catalog=glue_catalog and lakehouse_bucket=s3://dummy-lakehouse

## 2. Read Bronze

In [7]:
bronze_df = spark.read.option("mode", "FAILFAST").json(BRONZE_PATHS)
bronze_df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

200

In [8]:
print("Running Bronze validations...")

if bronze_df.count() == 0:
    raise RuntimeError("Bronze validation failed: no records found")

required_cols = {
    "symbol",
    "trade_date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "ingested_at"
}

missing = required_cols - set(bronze_df.columns)
if missing:
    raise RuntimeError(f"Bronze validation failed: missing columns {missing}")

print("Bronze validations passed")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running Bronze validations...
Bronze validations passed

## 3. Transform to Silver

In [9]:
silver_incoming = (
    bronze_df
    # ----------------------------
    # Business keys
    # ----------------------------
    .withColumn("symbol", col("symbol"))
    .withColumn("trade_date", to_date(col("trade_date")))

    # ----------------------------
    # OHLCV metrics
    # ----------------------------
    .withColumn("open", col("open").cast("double"))
    .withColumn("high", col("high").cast("double"))
    .withColumn("low", col("low").cast("double"))
    .withColumn("close", col("close").cast("double"))
    .withColumn("volume", col("volume").cast("bigint"))

    # ----------------------------
    # Source metadata
    # ----------------------------
    .withColumn("source", col("source"))
    .withColumn("dataset", col("dataset"))
    .withColumn("request_url", col("request_url"))

    # ----------------------------
    # Timestamps
    # ----------------------------
    .withColumn("received_at", to_timestamp(col("received_at")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at")))

    # ----------------------------
    # SCD2 record hash
    # ----------------------------
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("symbol"),
                col("trade_date").cast("string"),
                col("open"),
                col("high"),
                col("low"),
                col("close"),
                col("volume")
            ),
            256
        )
    )

    # ----------------------------
    # SCD2 control columns
    # ----------------------------
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))

    # ----------------------------
    # Lineage & orchestration
    # ----------------------------
    .withColumn("dag_id", lit(DAG_ID))
    .withColumn("run_id", lit(RUN_ID))
    .withColumn("processed_at", current_timestamp())
)

silver_incoming.createOrReplaceTempView("incoming")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 4. Deduplicate incoming batch

In [10]:
w = Window.partitionBy("symbol", "trade_date").orderBy(col("ingested_at").desc())
silver_dedup = (
  silver_incoming
  .withColumn("rn", row_number().over(w))
  .filter(col("rn") == 1)
  .drop("rn")
)

silver_dedup.createOrReplaceTempView("incoming_dedup")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
print("Running incoming batch validations...")

dup_cnt = spark.sql("""
SELECT COUNT(*) AS cnt
FROM (
  SELECT symbol, trade_date
  FROM incoming_dedup
  GROUP BY symbol, trade_date
  HAVING COUNT(*) > 1
)
""").collect()[0]["cnt"]

if dup_cnt > 0:
    raise RuntimeError(f"Incoming validation failed: {dup_cnt} duplicate business keys after deduplication")

print("Incoming validations passed")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running incoming batch validations...
Incoming validations passed

## 5. Merge Silver

In [26]:
spark.sql(f''' 
CREATE TABLE IF NOT EXISTS {SILVER_FQN} (
  symbol STRING,
  trade_date DATE,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE,
  volume BIGINT,
  source STRING,
  dataset STRING,
  request_url STRING,
  received_at TIMESTAMP,
  ingested_at TIMESTAMP,
  record_hash STRING,
  effective_from TIMESTAMP,
  effective_to TIMESTAMP,
  is_current BOOLEAN,
  dag_id STRING,
  run_id STRING,
  processed_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (trade_date);
''')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [17]:
spark.sql(f"""
MERGE INTO {SILVER_FQN} t
USING incoming_dedup s
ON t.symbol = s.symbol AND t.trade_date = s.trade_date AND t.is_current = true
WHEN MATCHED AND t.record_hash <> s.record_hash THEN
  UPDATE SET t.effective_to = s.effective_from, t.is_current = false
WHEN NOT MATCHED THEN INSERT *
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [18]:
print("Running Silver SCD2 invariant checks...")

multiple_currents = spark.sql(f"""
SELECT symbol, trade_date
FROM {SILVER_FQN}
WHERE is_current = true
GROUP BY symbol, trade_date
HAVING COUNT(*) > 1
""").count()

if multiple_currents > 0:
    raise ValueError(f"Found current records ({multiple_currents} records) for single business keys (symbol, trade_date) indicating FAULTY MERGE")

missing_currents = spark.sql(f"""
SELECT symbol, trade_date
FROM {SILVER_FQN}
GROUP BY symbol, trade_date
HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) = 0
""").count()

if missing_currents > 0:
    raise ValueError(f"Found records that are missing ({missing_currents} records) for single business keys (symbol, trade_date) indicating DATA LOSS and FAULTY MERGE")
    
print("Silver validations passed")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running Silver SCD2 invariant checks...
Silver validations passed

In [15]:
missing_currents = spark.sql(f"""
SELECT symbol, trade_date
FROM {SILVER_FQN}
GROUP BY symbol, trade_date
HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) = 0
""")

missing_currents.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+
|symbol|trade_date|
+------+----------+
+------+----------+

In [16]:
spark.sql(f"""
SELECT *
FROM {SILVER_FQN}
WHERE symbol = 'MSFT' and trade_date = '2025-12-10'
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+-----+------+------+------+--------+------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+----------+--------------------+--------------------+--------------------+
|symbol|trade_date| open|  high|   low| close|  volume|      source|          dataset|         request_url|         received_at|         ingested_at|         record_hash|      effective_from|effective_to|is_current|              dag_id|              run_id|        processed_at|
+------+----------+-----+------+------+------+--------+------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+----------+--------------------+--------------------+--------------------+
|  MSFT|2025-12-10|48.03|484.25|475.08|478.56|35756229|alphavantage|time_series_daily|https://www.alpha...|2025-12-19 18:58:...|2025-12-19 18:58:...|d4fd27d466a66e

In [13]:
spark.sql(f"""
SELECT * FROM {SILVER_FQN}.snapshots;
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-01-22 14:53:...|1578918455844481577|               NULL|   append|s3://dummy-lakeho...|{spark.app.id -> ...|
|2026-01-23 14:56:...|3880006026395862142|1578918455844481577|overwrite|s3://dummy-lakeho...|{spark.app.id -> ...|
|2026-01-23 16:33:...|4581394440506736971|3880006026395862142|overwrite|s3://dummy-lakeho...|{spark.app.id -> ...|
|2026-01-23 16:35:...|7918620556970773282|4581394440506736971|overwrite|s3://dummy-lakeho...|{spark.app.id -> ...|
|2026-01-23 16:39:...|1797537165395748007|7918620556970773282|overwrite|s3://dummy-lakeho...|{spark.app.id -> ...|
|2026-01-24 07:09:...|3429322716571457496|1578918455844481577|overwrite|s3://dum

In [14]:
spark.sql(f"""
CALL {GLUE_CATALOG}.system.rollback_to_snapshot(
  '{SILVER_DB}.{SILVER_TABLE}',
  1578918455844481577
)
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 3429322716571457496|1578918455844481577|
+--------------------+-------------------+

In [16]:
spark.sql(f"""
UPDATE {SILVER_FQN}
SET
  is_current     = true,
  effective_to   = NULL,
  effective_from = TIMESTAMP '2026-01-22 14:51:00'
WHERE symbol        = 'MSFT'
  AND trade_date    = '2025-12-10'
  AND is_current    = false
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

## 6. Rebuild Gold

In [20]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_FQN} (
  symbol STRING,
  trade_date DATE,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE,
  volume BIGINT,
  processed_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (trade_date)
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [21]:
spark.sql(f"""
INSERT OVERWRITE {GOLD_FQN}
SELECT
    symbol,
    trade_date,

    open,
    high,
    low,
    close,
    volume,

    current_timestamp() AS processed_at
FROM {SILVER_FQN}
WHERE is_current = true
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [22]:
silver_df = spark.read.table(SILVER_FQN)
silver_df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

200

In [23]:
gold_df = spark.read.table(GOLD_FQN)
gold_df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

200

In [9]:
gold_df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+-------+------+------+------+--------+--------------------+
|symbol|trade_date|   open|  high|   low| close|  volume|        processed_at|
+------+----------+-------+------+------+------+--------+--------------------+
|  AAPL|2026-01-02|272.255|277.84| 269.0|271.01|37838054|2026-01-21 05:54:...|
|  MSFT|2026-01-02|484.385|484.66|470.16|472.94|25571567|2026-01-21 05:54:...|
|  AAPL|2026-01-07|  263.2|263.68|259.81|260.33|48309804|2026-01-21 05:54:...|
|  MSFT|2026-01-07|479.755| 489.7|477.95|483.47|25564196|2026-01-21 05:54:...|
|  AAPL|2026-01-08| 257.02|259.29| 255.7|259.04|50419337|2026-01-21 05:54:...|
+------+----------+-------+------+------+------+--------+--------------------+
only showing top 5 rows

In [24]:
gold_df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+-------+------+------+------+---------+--------------------+
|symbol|trade_date|   open|  high|   low| close|   volume|        processed_at|
+------+----------+-------+------+------+------+---------+--------------------+
|  AAPL|2025-07-31| 208.49|209.84|207.16|207.57| 80698431|2026-01-22 15:12:...|
|  MSFT|2025-07-31|555.225|555.45| 531.9| 533.5| 51617326|2026-01-22 15:12:...|
|  AAPL|2025-08-01|210.865|213.58| 201.5|202.38|104434473|2026-01-22 15:12:...|
|  MSFT|2025-08-01|  535.0| 535.8|520.86|524.11| 28977628|2026-01-22 15:12:...|
|  AAPL|2025-07-30|211.895|212.39|207.72|209.05| 45512514|2026-01-22 15:12:...|
+------+----------+-------+------+------+------+---------+--------------------+
only showing top 5 rows

In [25]:
print(GOLD_FQN)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

glue_catalog.gold.american_markets_alphavantage_time_series_daily_fact

## Metrics

In [27]:
daily_window = Window.partitionBy("symbol").orderBy("trade_date")

rolling_7 = daily_window.rowsBetween(-6, 0)
rolling_30 = daily_window.rowsBetween(-29, 0)

monthly_window = Window.partitionBy(
    "symbol",
    year("trade_date"),
    month("trade_date")
)

yearly_window = Window.partitionBy(
    "symbol",
    year("trade_date")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
gold_fact_df = spark.table(GOLD_FQN)

df_metrics = (
    gold_fact_df
    .withColumn(
        "daily_return",
        (col("close") - lag("close").over(daily_window)) /
        lag("close").over(daily_window)
    )
    .withColumn(
        "ma_7_close",
        avg("close").over(rolling_7)
    )
    .withColumn(
        "ma_30_close",
        avg("close").over(rolling_30)
    )
    .withColumn(
        "volatility_30",
        stddev("daily_return").over(rolling_30)
    )
    .withColumn(
        "monthly_avg_close",
        avg("close").over(monthly_window)
    )
    .withColumn(
        "yearly_avg_close",
        avg("close").over(yearly_window)
    )
    .withColumn(
        "processed_at",
        current_timestamp()
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
df_metrics.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+-------+------+--------+------+---------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+
|symbol|trade_date|   open|  high|     low| close|   volume|        processed_at|        daily_return|        ma_7_close|       ma_30_close|       volatility_30|monthly_avg_close|  yearly_avg_close|
+------+----------+-------+------+--------+------+---------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+
|  AAPL|2025-07-30|211.895|212.39|  207.72|209.05| 45512514|2026-01-22 16:10:...|                NULL|            209.05|            209.05|                NULL|           208.31|252.24509999999998|
|  AAPL|2025-07-31| 208.49|209.84|  207.16|207.57| 80698431|2026-01-22 16:10:...|-0.00707964601769...|            208.31|            208.31|                NULL|           208.31|252.24509999999998|
|  AA

In [35]:
df_metrics.write.mode("overwrite").format("parquet").saveAsTable(METRICS_FQN)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [37]:
df_metrics.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

200

In [7]:
df_metrics_table = spark.read.option("mode", "FAILFAST").table(METRICS_FQN)
df_metrics_table.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+----------+-------+------+--------+------+---------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+
|symbol|trade_date|   open|  high|     low| close|   volume|        processed_at|        daily_return|        ma_7_close|       ma_30_close|       volatility_30|monthly_avg_close|  yearly_avg_close|
+------+----------+-------+------+--------+------+---------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+
|  AAPL|2025-07-30|211.895|212.39|  207.72|209.05| 45512514|2026-01-22 16:40:...|                NULL|            209.05|            209.05|                NULL|           208.31|252.24509999999998|
|  AAPL|2025-07-31| 208.49|209.84|  207.16|207.57| 80698431|2026-01-22 16:40:...|-0.00707964601769...|            208.31|            208.31|                NULL|           208.31|252.24509999999998|
|  AA

In [26]:
spark.sql(f"""
DROP TABLE {METRICS_FQN}
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [33]:
silver_df_dummy.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

200